# Import

In [173]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random
import ast

In [174]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Prep

## Loading variables

In [235]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")

In [236]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [237]:
# Loading result graph and communities
with open(f"{RESULT_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{RESULT_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{RESULT_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

In [238]:
for c in communities:
    print(len(c))

3245
3098
3010
2721
2657
2485
2454
690
546
402
258
43
20
16
15
13
13
11
9
7
5
5
5
4
3
3
3
3
3
2
2
2
2
2
2
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1


## Helpful functions (big object, drop NAN)

In [239]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to NCBI

In [240]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi_dict = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi_dict.get, c)) for c in comms]
    return comms_ncbi

In [241]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))
with open(f"{RESULT_FOLDER}/result_communities_ncbi_selected.pkl", "wb") as f:
    pickle.dump(communities_ncbi, f)

[['9522', '163486', '163590', '57403', '55754', '1192', '11252', '84162', '54509', '10636', '1182', '23637', '91966', '9236', '57222', '2803', '8417', '7993', '9202', '3613', '4212', '51199', '259230', '162427', '79956', '11078', '26353', '10807', '55317', '2029', '51136', '54332', '23190', '8444', '57465', '10421', '1620', '64783', '60592', '26268', '116068', '84919', '310', '64114', '23185', '11240', '53373', '83483', '23197', '80829', '4430', '91746', '7871', '57798', '23151', '81545', '54464', '3964', '10169', '54468', '4291', '5874', '29969', '79016', '84640', '51119', '4072', '54556', '10950', '83786', '23522', '81566', '23392', '5189', '139341', '51809', '10927', '9679', '23621', '9898', '23051', '9847', '57211', '10391', '57706', '7105', '29097', '9980', '1983', '22992', '23325', '23274', '7289', '54751', '56270', '23177', '148789', '27072', '27147', '10788', '26090', '23673', '57406', '9828', '79848', '253782', '9559', '55745', '56983', '22979', '54664', '63898', '9903', '1521

In [242]:
communities_ncbi_full = index_to_ncbi(communities,index_to_gene_distinct)
print(communities_ncbi_full)
print(len(communities_ncbi_full))
with open(f"{RESULT_FOLDER}/result_communities_ncbi.pkl", "wb") as f:
    pickle.dump(communities_ncbi_full, f)

[['15', '18', '26', '29', '33', '35', '36', '37', '41', '47', '50', '55', '89', '113', '125', '127', '128', '130', '131', '157', '158', '160', '161', '163', '164', '177', '186', '191', '197', '203', '204', '210', '211', '212', '213', '214', '215', '216', '218', '225', '231', '239', '242', '246', '262', '276', '277', '278', '285', '290', '291', '310', '311', '316', '325', '341', '350', '359', '362', '366', '375', '383', '395', '396', '397', '399', '406', '429', '435', '467', '476', '478', '482', '490', '501', '509', '513', '515', '521', '522', '523', '525', '527', '537', '539', '549', '550', '570', '575', '580', '583', '586', '587', '593', '594', '610', '635', '636', '643', '653', '660', '661', '668', '669', '687', '695', '699', '715', '722', '725', '728', '735', '752', '760', '762', '765', '778', '779', '780', '784', '790', '799', '815', '819', '823', '824', '835', '843', '844', '859', '860', '873', '875', '887', '896', '899', '912', '916', '917', '923', '930', '931', '933', '939', '94

## NCBI to HGNC

In [243]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        symbols = [ncbi_to_hgnc_dict.get(n) for n in community]
        comms_HGNC.append(symbols)
    return comms_HGNC

In [244]:
# # NCBI to HGNC symbol
# def ncbi_to_HGNC(comms_ncbi):
#     comms_HGNC = []
#     for community in comms_ncbi:
#         mg = mygene.MyGeneInfo()
#         entrez_ids = [str(e) for e in community]

#         results = mg.querymany(
#             entrez_ids,
#             scopes="entrezgene",
#             fields="symbol",
#             species="human"
#         )

#         # Build a mapping: input ID -> symbol (or None)
#         id_to_symbol = {}
#         for r in results:
#             q = str(r.get("query"))
#             id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

#         # Preserve original order
#         symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
#         comms_HGNC.append(symbols)
#     return comms_HGNC


In [245]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)
COMMUNITIES_HGNC_full = ncbi_to_HGNC(communities_ncbi_full)

In [246]:
print(len(COMMUNITIES_HGNC))

12


In [247]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)
COMMUNITIES_HGNC_full = drop_nan_from_communities(COMMUNITIES_HGNC_full)

Community 0: dropped 0 NaN entries
Community 1: dropped 9 NaN entries
Community 2: dropped 0 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 4 NaN entries
Community 5: dropped 0 NaN entries
Community 6: dropped 0 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 0 NaN entries
Community 9: dropped 0 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries

Total dropped across all communities: 13
Community 0: dropped 0 NaN entries
Community 1: dropped 28 NaN entries
Community 2: dropped 0 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 8 NaN entries
Community 5: dropped 1 NaN entries
Community 6: dropped 0 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 1 NaN entries
Community 9: dropped 0 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 0 NaN entries
Community 13: dropped 0 NaN entries
Community 14: dropped 0 NaN entries
Commu

In [248]:
with open(f"{RESULT_FOLDER}/result_communities_HGNC_selected.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC, f)
with open(f"{RESULT_FOLDER}/result_communities_HGNC.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC_full, f)

In [249]:
print(len(COMMUNITIES_HGNC))
print(len(COMMUNITIES_HGNC_full))

12
262


# Categoization Prep

### GO-slim

In [162]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [163]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [164]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [165]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [166]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [167]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [168]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=300)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

In [169]:
# Specific for Reactome: build level map first
reactome_level1 = build_reactome_level_map(level = 1)

https://reactome.org/ContentService/data/eventsHierarchy/9606


# Run Enrichment Analysis

In [170]:
TERM_SCORE_CAP = 0.001
PERCENTAGE = 0.1

### GO

In [171]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        

        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["GO_ID"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["GO_ID"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [172]:
go_important_terms = go_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE,slim_ids,depth = 1)

Size of community: 1097
Number of filtered terms: 68
Number of unmapped terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_68820\3881098508.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
38,0,Negative Regulation Of Cilium Assembly (GO:1902018),7/14,3.112898e-04,{GO:0065007},[biological regulation]
46,0,Reticulophagy (GO:0061709),7/16,7.872433e-04,{GO:0009987},[cellular process]
36,0,Protein K48-linked Deubiquitination (GO:0071108),9/24,2.484139e-04,{GO:0009987},[cellular process]
3997,0,Cul3-RING Ubiquitin Ligase Complex (GO:0031463),12/35,9.522100e-06,{GO:0032991},[protein-containing complex]
39,0,Regulation Of Cell Morphogenesis (GO:0022604),10/31,3.112898e-04,{GO:0065007},[biological regulation]
10,0,Regulation Of TORC1 Signaling (GO:1903432),16/50,1.573329e-06,{GO:0065007},[biological regulation]
3431,0,Phosphatidylinositol-3-Phosphate Binding (GO:0032266),12/40,4.119084e-05,{GO:0005488},[binding]
49,0,Negative Regulation Of BMP Signaling Pathway (GO:0030514),11/43,9.976708e-04,{GO:0065007},[biological regulation]
3424,0,Cysteine-Type Deubiquitinase Activity (GO:0004843),25/98,6.355289e-09,{GO:0003824},[catalytic activity]
24,0,Endocytic Recycling (GO:0032456),16/64,3.321460e-05,"{GO:0051179, GO:0009987}","[localization, cellular process]"


Size of community: 1140
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
1072,1,RNA Polymerase II Cis-Regulatory Region Sequence-Specific DNA Binding (GO:0000978),114/1122,1.201728e-07,{GO:0005488},[binding]
1071,1,RNA Polymerase II Transcription Regulatory Region Sequence-Specific DNA Binding (GO:0000977),124/1225,5.169911e-08,{GO:0005488},[binding]
1073,1,Cis-Regulatory Region Sequence-Specific DNA Binding (GO:0000987),111/1098,1.811394e-07,{GO:0005488},[binding]


...

Size of community: 848
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
2,5,Protein Insertion Into ER Membrane By Stop-Transfer Membrane-Anchor Sequence (GO:0045050),7/9,7.203307e-06,"{GO:0051179, GO:0009987}","[localization, cellular process]"
1,5,Tail-Anchored Membrane Protein Insertion Into ER Membrane (GO:0071816),11/16,3.612776e-09,"{GO:0051179, GO:0009987}","[localization, cellular process]"
0,5,Protein Insertion Into ER Membrane (GO:0045048),14/29,3.612776e-09,"{GO:0051179, GO:0009987}","[localization, cellular process]"


Size of community: 848
Number of filtered terms: 6
Number of unmapped terms: 1


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
2087,6,Azurophil Granule (GO:0042582),20/155,3.177660e-04,{GO:0110165},[cellular anatomical structure]
2081,6,Collagen-Containing Extracellular Matrix (GO:0062023),45/373,6.604140e-08,{},[]
2082,6,Actin Cytoskeleton (GO:0015629),36/327,1.615538e-05,{GO:0110165},[cellular anatomical structure]
2085,6,Endoplasmic Reticulum Lumen (GO:0005788),31/284,7.185172e-05,{GO:0110165},[cellular anatomical structure]
2083,6,Focal Adhesion (GO:0005925),40/387,1.615538e-05,{GO:0110165},[cellular anatomical structure]
2084,6,Cell-Substrate Junction (GO:0030055),40/395,2.076184e-05,{GO:0110165},[cellular anatomical structure]


KeyboardInterrupt: 

In [ ]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value)
0,0,1540,COPI Vesicle Coat (GO:0030126),7/12,6.412616e-04,[protein-containing complex],GO_Cellular_Component_2023,8.875593e-06,0.0,0.0,16.853881,1.960478e+02,COPA;COPB1;TMED3;COPZ2;COPG1;TMED7;COPE,GO:0030126,{GO:0032991},0.583333
1,0,1540,Small-Subunit Processome (GO:0032040),22/73,2.063188e-06,[protein-containing complex],GO_Cellular_Component_2023,1.531107e-08,0.0,0.0,5.231316,9.413590e+01,NOP56;WDR36;NOP14;KRR1;PNO1;UTP3;WDR3;MRPS12;F...,GO:0032040,{GO:0032991},0.301370
2,0,1540,Ribosomal Small Subunit Biogenesis (GO:0042274),21/84,7.104291e-04,[cellular process],GO_Biological_Process_2023,1.056662e-06,0.0,0.0,4.037086,5.555190e+01,NOP56;WDR36;NOP14;KRR1;PNO1;UTP3;WDR3;MRPS12;F...,GO:0042274,{GO:0009987},0.250000
3,0,1540,rRNA Processing (GO:0006364),24/101,5.094965e-04,[cellular process],GO_Biological_Process_2023,5.052023e-07,0.0,0.0,3.779529,5.479677e+01,NOP56;NOP14;WDR36;DDX27;WDR3;MAK16;DDX10;DDX52...,GO:0006364,{GO:0009987},0.237624
4,0,1540,Ribosome Biogenesis (GO:0042254),32/155,4.632809e-04,[cellular process],GO_Biological_Process_2023,2.296881e-07,0.0,0.0,3.163529,4.835942e+01,DDX27;WDR3;NIP7;MRPS12;FCF1;WDR46;MRM2;NOL6;RR...,GO:0042254,{GO:0009987},0.206452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
864,8,275,Detection Of Chemical Stimulus Involved In Sen...,86/139,9.420555e-128,[response to stimulus],GO_Biological_Process_2023,1.662451e-128,0.0,0.0,168.892083,4.969187e+04,OR10J1;OR2A1;OR4E2;OR10J3;OR4E1;OR10J5;OR10J4;...,GO:0050911,{GO:0050896},0.618705
865,8,275,Detection Of Chemical Stimulus Involved In Sen...,87/141,5.591666e-129,[response to stimulus],GO_Biological_Process_2023,6.578431e-130,0.0,0.0,168.575355,5.014312e+04,OR10J1;OR2A1;OR4E2;OR10J3;OR4E1;OR10J5;OR10J4;...,GO:0050907,{GO:0050896},0.617021
866,8,275,Olfactory Receptor Activity (GO:0004984),220/362,0.000000e+00,[molecular transducer activity],GO_Molecular_Function_2023,0.000000e+00,0.0,0.0,551.633803,inf,OR7G2;OR52N1;OR2M5;OR2M4;OR10AC1;OR51L1;OR2AE1...,GO:0004984,{GO:0060089},0.607735
867,8,275,Sensory Perception Of Smell (GO:0007608),137/230,1.431229e-206,[multicellular organismal process],GO_Biological_Process_2023,8.418996e-208,0.0,0.0,209.567087,9.992310e+04,OR1C1;OR2M5;OR2M4;OR2T12;OR2T10;OR10AC1;OR2T11...,GO:0007608,{GO:0032501},0.595652


### KEGG

In [ ]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [ ]:
kegg_important_terms = kegg_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE)

Size of community: 1206
Number of filtered terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_68820\1053027199.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,1,Herpes simplex virus 1 infection,61/498,0.000006,hsa05168,[Infectious disease: viral]


Size of community: 1270
Number of filtered terms: 6


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
2,2,Mucin type O-glycan biosynthesis,13/36,1.073805e-05,hsa00512,[Glycan biosynthesis and metabolism]
1,2,N-Glycan biosynthesis,17/50,5.694808e-07,hsa00510,[Glycan biosynthesis and metabolism]
3,2,Sphingolipid metabolism,15/49,1.104005e-05,hsa00600,[Lipid metabolism]
4,2,Glycosphingolipid biosynthesis,12/45,7.100978e-04,NaN,[]
5,2,Glycosaminoglycan biosynthesis,13/53,7.100978e-04,NaN,[]
0,2,RNA transport,37/186,9.384303e-08,NaN,[]


...

Size of community: 543
Number of filtered terms: 1


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,6,Neuroactive ligand-receptor interaction,68/341,3.017443e-37,hsa04080,[Signaling molecules and interaction]


Size of community: 275
Number of filtered terms: 1


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,8,Olfactory transduction,266/440,0.0,hsa04740,[Sensory system]


5 out of 10 communities had significant GO terms.


In [ ]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,KEGG_ID,Overlap (value)
0,1,1206,Herpes simplex virus 1 infection,61/498,5.600529e-06,[Infectious disease: viral],KEGG_2021_Human,1.056704e-07,0.0,0.0,2.237920,35.947583,ZNF133;ZNF573;ZNF253;ZNF250;ZFP82;ZNF83;ZSCAN3...,hsa05168,0.122490
1,2,1270,Mucin type O-glycan biosynthesis,13/36,1.073805e-05,[Glycan biosynthesis and metabolism],KEGG_2021_Human,1.498332e-07,0.0,0.0,8.411712,132.179478,GALNT12;GALNT7;GALNT11;GALNT6;GALNT14;GALNT5;S...,hsa00512,0.361111
2,2,1270,N-Glycan biosynthesis,17/50,5.694808e-07,[Glycan biosynthesis and metabolism],KEGG_2021_Human,5.297496e-09,0.0,0.0,7.686982,146.483364,ST6GAL2;ALG5;ALG13;ALG3;MOGS;ALG10;FUT8;GANAB;...,hsa00510,0.340000
3,2,1270,Sphingolipid metabolism,15/49,1.104005e-05,[Lipid metabolism],KEGG_2021_Human,2.053963e-07,0.0,0.0,6.572299,101.202394,CERS3;CERS4;CERS6;CERK;SGMS1;SPHK2;SGPP2;NEU3;...,hsa00600,0.306122
4,2,1270,Glycosphingolipid biosynthesis,12/45,7.100978e-04,[],KEGG_2021_Human,1.652113e-05,0.0,0.0,5.404538,59.508669,B3GALNT1;ST8SIA1;B3GALT4;B3GNT3;B3GNT2;B4GALNT...,NaN,0.266667
5,2,1270,Glycosaminoglycan biosynthesis,13/53,7.100978e-04,[],KEGG_2021_Human,1.981668e-05,0.0,0.0,4.832339,52.329333,HS3ST3B1;GLCE;CSGALNACT2;EXTL2;FUT8;CHST11;DSE...,NaN,0.245283
6,2,1270,RNA transport,37/186,9.384303e-08,[],KEGG_2021_Human,4.364792e-10,0.0,0.0,3.742152,80.651916,NUP205;DDX20;SUMO4;NMD3;NXF1;EIF2B1;RPP14;RAE1...,NaN,0.198925
7,4,1158,Spliceosome,69/150,9.092484e-43,[Transcription],KEGG_2021_Human,3.852747e-45,0.0,0.0,14.675475,1500.824796,RBM25;EIF4A3;HNRNPU;PRPF19;PQBP1;EFTUD2;SNRPD2...,hsa03040,0.460000
8,4,1158,RNA polymerase,12/31,1.598606e-06,[Transcription],KEGG_2021_Human,6.773754e-08,0.0,0.0,10.373657,171.244436,POLR2B;POLR2C;POLR2D;POLR2E;POLR2F;POLR3G;POLR...,hsa03020,0.387097
9,4,1158,RNA degradation,24/79,4.959169e-10,"[Folding, sorting and degradation]",KEGG_2021_Human,8.405371e-12,0.0,0.0,7.229245,184.361288,HSPA9;BTG2;BTG1;PNPT1;ENO1;ENO2;LSM5;TOB1;LSM4...,hsa03018,0.303797


### Reactome

In [ ]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [ ]:
reactome_important_terms = reactome_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE)

Size of community: 1540
Number of filtered terms: 7


C:\Users\celem\AppData\Local\Temp\ipykernel_68820\3965740400.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
4,0,rRNA Modification In Nucleus And Cytosol R-HSA-6790901,22/60,3.402811e-08,[Metabolism of RNA]
0,0,rRNA Processing R-HSA-72312,50/199,3.078299e-11,[Metabolism of RNA]
2,0,Major Pathway Of rRNA Processing In Nucleolus And Cytosol R-HSA-6791226,44/179,7.745997e-10,[Metabolism of RNA]
3,0,rRNA Processing In Nucleus And Cytosol R-HSA-8868773,45/189,1.079243e-09,[Metabolism of RNA]
6,0,CDC42 GTPase Cycle R-HSA-9013148,28/149,9.437688e-04,[Signal Transduction]
1,0,Metabolism Of RNA R-HSA-8953854,105/666,3.957237e-10,[Metabolism of RNA]
5,0,RHO GTPase Cycle R-HSA-9012999,62/441,3.546184e-04,[Signal Transduction]


Size of community: 1206
Number of filtered terms: 2


,Community Index,Term,Overlap,Adjusted P-value,Category
1,1,Formation Of Cornified Envelope R-HSA-6809371,21/74,1.184298e-07,[Developmental Biology]
0,1,Keratinization R-HSA-6805567,50/208,2.648266e-15,[Developmental Biology]


...

Size of community: 543
Number of filtered terms: 15


,Community Index,Term,Overlap,Adjusted P-value,Category
8,6,Lysosphingolipid And LPA Receptors R-HSA-419408,10/14,5.703192e-12,[Signal Transduction]
16,6,Relaxin Receptors R-HSA-444821,5/8,1.257375e-05,[Signal Transduction]
12,6,Nucleotide-like (Purinergic) Receptors R-HSA-418038,8/16,6.464057e-08,[Signal Transduction]
15,6,P2Y Receptors R-HSA-417957,6/12,5.521293e-06,[Signal Transduction]
0,6,Class A/1 (Rhodopsin-like Receptors) R-HSA-373076,84/327,1.544547e-55,[Signal Transduction]
4,6,Peptide Ligand-Binding Receptors R-HSA-375276,50/196,1.115220e-32,[Signal Transduction]
2,6,GPCR Ligand Binding R-HSA-500792,93/458,1.543934e-52,[Signal Transduction]
5,6,G Alpha (Q) Signaling Events R-HSA-416476,43/212,1.096878e-23,[Signal Transduction]
14,6,Chemokine Receptors Bind Chemokines R-HSA-380108,11/56,4.945093e-06,[Signal Transduction]
7,6,ADORA2B Mediated Anti-Inflammatory Cytokine Production R-HSA-9660821,24/131,4.092912e-12,[Disease]


Size of community: 275
Number of filtered terms: 3


,Community Index,Term,Overlap,Adjusted P-value,Category
2,8,Expression And Translocation Of Olfactory Receptors R-HSA-9752946,258/393,0.0,[Sensory Perception]
1,8,Olfactory Signaling Pathway R-HSA-381753,258/401,0.0,[Sensory Perception]
0,8,Sensory Perception R-HSA-9709957,258/616,0.0,[Sensory Perception]


7 out of 10 communities had significant GO terms.


In [ ]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1540,rRNA Modification In Nucleus And Cytosol R-HSA...,22/60,3.402811e-08,[Metabolism of RNA],Reactome_2022,2.241641e-10,0.0,0.0,7.025934,156.106726,NOP56;WDR36;NOP14;KRR1;PNO1;UTP3;WDR3;FCF1;IMP...,0.366667
1,0,1540,rRNA Processing R-HSA-72312,50/199,3.078299e-11,[Metabolism of RNA],Reactome_2022,4.055730e-14,0.0,0.0,4.123913,127.165242,RBM28;RPL3;WDR3;RPLP0;FCF1;THUMPD1;WDR46;NOB1;...,0.251256
2,0,1540,Major Pathway Of rRNA Processing In Nucleolus ...,44/179,7.745997e-10,[Metabolism of RNA],Reactome_2022,3.061659e-12,0.0,0.0,3.992375,105.846095,RBM28;RPL3;WDR3;NIP7;RPL12;RPLP0;FCF1;ISG20L2;...,0.245810
3,0,1540,rRNA Processing In Nucleus And Cytosol R-HSA-8...,45/189,1.079243e-09,[Metabolism of RNA],Reactome_2022,5.687710e-12,0.0,0.0,3.828595,99.132721,RBM28;RPL3;WDR3;NIP7;RPL12;RPLP0;FCF1;ISG20L2;...,0.238095
4,0,1540,CDC42 GTPase Cycle R-HSA-9013148,28/149,9.437688e-04,[Signal Transduction],Reactome_2022,8.704060e-06,0.0,0.0,2.806703,32.702925,CPNE8;FAM13B;WIPF1;DOCK9;SNAP23;ARHGAP5;IQGAP3...,0.187919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,6,543,G Alpha (I) Signaling Events R-HSA-418594,46/312,2.176741e-19,[Signal Transduction],Reactome_2022,5.403257e-21,0.0,0.0,6.677554,311.623318,RGS18;GNAZ;RGS17;SUCNR1;RGSL1;PMCH;RXFP4;RRH;L...,0.147436
205,6,543,Anti-inflammatory Response Favoring Leishmania...,24/165,5.217359e-10,[Disease],Reactome_2022,2.035140e-11,0.0,0.0,6.334932,155.952545,GPR27;GNAZ;GPR39;GPR25;VIPR2;CALCB;GPR45;SCT;G...,0.145455
206,8,275,Expression And Translocation Of Olfactory Rece...,258/393,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,2202.274510,inf,OR7G2;OR11H2;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;...,0.656489
207,8,275,Olfactory Signaling Pathway R-HSA-381753,258/401,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,2078.221308,inf,OR7G2;OR11H2;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;...,0.643392


### Disease Data Sets

In [ ]:
# disease_term_score_cap = 0.001
# disease_percentage = 0.1
# important_diseases = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value"])

In [ ]:
# # Disease-gene enrichment libraries
# disease_sets = [
#     'DisGeNET_2020', # curated gene–disease associations
#     'GWAS_Catalog_2023', # genome-wide association hits
#     'OMIM_Disease', # Mendelian disorders
#     'Jensen_DISEASES' # text-mined associations
# ]

# # # Disease-gene enrichment Analysis; save terms with small size and high p-value
# i = 0
# for community in communities_HGNC:
#     # Gene Ontology enrichment
#     enr_disease = gp.enrichr(
#         gene_list=community,
#         gene_sets=disease_sets,
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     enr_disease_df = enr_disease.results.sort_values('Adjusted P-value')
#     print(f"Size of community: {len(community)}")

#     mask =  (enr_disease_df["Adjusted P-value"] < disease_term_score_cap) & (enr_disease_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > disease_percentage))
        
#     filtered = enr_disease_df[mask].copy()
#     if not filtered.empty:
#         filtered.loc[:, "Community Index"] = i
#         filtered.loc[:, "Community Size"] = len(community)
#         important_diseases = pd.concat([important_diseases, filtered], ignore_index=True)

#     display(HTML(filtered[['Term','Overlap','Adjusted P-value']].head(10).to_html(max_cols=None)))
#     i += 1

# Important Terms df

In [ ]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)
important_terms = important_terms.sort_values(by="Community Index")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value),KEGG_ID
0,1,1241,THO Complex Part Of Transcription Export Compl...,5/5,2.282035e-05,[protein-containing complex],GO_Cellular_Component_2023,9.128139e-07,0.0,0.0,93795.000000,1.304382e+06,THOC3;THOC2;THOC5;THOC7;THOC6,GO:0000445,{GO:0032991},1.000000,NaN
895,1,1241,Post-translational Protein Modification R-HSA-...,152/1383,5.215519e-10,[Metabolism of proteins],Reactome_2022,1.678815e-12,0.0,0.0,1.987422,5.388483e+01,GALNT12;GALNT11;GALNT14;KDELR1;GALNT18;SMC6;UB...,NaN,NaN,0.109906,NaN
70,1,1241,Nucleolus (GO:0005730),80/771,7.440910e-05,[cellular anatomical structure],GO_Cellular_Component_2023,4.063280e-06,0.0,0.0,1.801730,2.236582e+01,MAPKBP1;DOCK4;TUT4;PAK1IP1;PRDM5;CTCF;NOC2L;MY...,GO:0005730,{GO:0110165},0.103761,NaN
69,1,1241,Nuclear Lumen (GO:0031981),81/780,6.961100e-05,[cellular anatomical structure],GO_Cellular_Component_2023,3.427003e-06,0.0,0.0,1.804129,2.270284e+01,MAPKBP1;DOCK4;TUT4;PAK1IP1;PRDM5;CTCF;NOC2L;MY...,GO:0031981,{GO:0110165},0.103846,NaN
68,1,1241,RNA Binding (GO:0003723),150/1411,2.054331e-08,[binding],GO_Molecular_Function_2023,3.021075e-11,0.0,0.0,1.907831,4.621305e+01,OTUD4;TCERG1;CISD2;NOC2L;RRP8;RRP9;LSM10;EIF4E...,GO:0003723,{GO:0005488},0.106308,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842,9,144,Cornified Envelope (GO:0001533),8/41,3.093648e-09,[cellular anatomical structure],GO_Cellular_Component_2023,4.640472e-10,0.0,0.0,35.335116,7.593882e+02,SPRR2F;SPRR2G;TCHH;SPRR2A;DSG4;SPRR2B;DSC1;SPRR2D,GO:0001533,{GO:0110165},0.195122,NaN
841,9,144,Keratin Filament (GO:0045095),12/39,8.056405e-16,[cellular anatomical structure],GO_Cellular_Component_2023,4.028203e-17,0.0,0.0,66.764310,2.520395e+03,KRT82;KRT36;KRT79;KRT78;KRT77;KRT76;KRT86;KRT7...,GO:0045095,{GO:0110165},0.307692,NaN
840,9,144,Intermediate Filament Organization (GO:0045109),25/68,2.978273e-35,[cellular process],GO_Biological_Process_2023,6.204736e-37,0.0,0.0,96.799883,8.070239e+03,KRT82;KRT24;TCHH;KRT86;KRT85;KRT40;KRT84;KRT33...,GO:0045109,{GO:0009987},0.367647,NaN
873,9,144,Staphylococcus aureus infection,13/95,6.788863e-13,[Infectious disease: bacterial],KEGG_2021_Human,1.697216e-13,0.0,0.0,23.930553,7.036687e+02,KRT24;KRT35;KRT32;KRT31;KRT40;KRT33A;KRT28;KRT...,NaN,NaN,0.136842,hsa05150


In [ ]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms.csv", index=False)

# Robustness Analysis

In [ ]:
# def run_enrichment_func(community,term_score_cap,percentage):
#     # GO df
#     enr_go = gp.enrichr(
#         gene_list=community,
#         gene_sets=['GO_Biological_Process_2023',
#                 'GO_Molecular_Function_2023',
#                 'GO_Cellular_Component_2023'],
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     GO_df = enr_go.results
#     mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     GO_df = GO_df[mask].copy()   
    
#     # KEGG df
#     enr_kegg = gp.enrichr(
#         gene_list=community,
#         gene_sets=['KEGG_2021_Human'],
#         organism='Human',
#         outdir=None
#     )
#     KEGG_df = enr_kegg.results
#     mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     KEGG_df = KEGG_df[mask].copy() 
       
#     # Reactome df
#     enr_reactome = gp.enrichr(
#         gene_list=community,
#         gene_sets=['Reactome_2022'],
#         organism='Human',
#         outdir=None
#     )
#     Reactome_df = enr_reactome.results  
#     mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     Reactome_df = Reactome_df[mask].copy()
    
    
#     all_df = [GO_df,KEGG_df,Reactome_df]
#     # build result df by concatenating
#     result = pd.concat(all_df, ignore_index=True)
#     return result

In [ ]:
# from json import JSONDecodeError

# # ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
# _ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

# def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
#     """
#     Calls user's run_enrichment_func(community) with retries + memoization.
#     Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
#     """
#     # Ensure we always pass a list of gene symbols (never a bare string)
#     genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
#     if len(genes) == 0:
#         return pd.DataFrame()

#     key = tuple(sorted(genes))
#     if key in _ENR_CACHE:
#         return _ENR_CACHE[key].copy()

#     for a in range(retries):
#         try:
#             df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
#             if df is None:
#                 # treat as transient failure to trigger retry
#                 raise RuntimeError("run_enrichment_func returned None")
#             _ENR_CACHE[key] = df.copy()
#             return df
#         except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
#             # Transient errors from HTTP/JSON/file handling inside gseapy
#             if a == retries - 1:
#                 # Give up: return empty so pipeline continues
#                 return pd.DataFrame()
#             time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

#     return pd.DataFrame()

# # ---------------- 2) Minimal bootstrap to record robust terms ----------------
# def get_robust_terms(communities_HGNC, run_enrichment_func,
#                      R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
#     """
#     Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
#     Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
#     """
#     rng = np.random.default_rng(seed)
#     rows = []

#     for cid, community in enumerate(communities_HGNC):
#         n = len(community)
#         if n == 0:
#             continue
#         drop_k = max(1, int(np.floor(leaveout * n)))
#         counts = Counter()

#         for _ in range(R):
#             # Jackknife subset (ensure not empty)
#             keep = np.ones(n, dtype=bool)
#             keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
#             sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
#             if len(sub) == 0:
#                 continue

#             df = run_enrichment_safe(run_enrichment_func, sub)
#             if df is None or df.empty:
#                 continue

#             # Your function already returns significant terms; just count them.
#             # If it includes multiple libraries, preserve Gene_set to disambiguate names.
#             if 'Term' not in df.columns:
#                 continue  # be defensive

#             if 'Gene_set' in df.columns:
#                 terms = (df[['Term', 'Gene_set']]
#                          .dropna()
#                          .drop_duplicates()
#                          .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
#                          .tolist())
#             else:
#                 terms = df['Term'].dropna().drop_duplicates().tolist()

#             counts.update(terms)

#             # tiny pause helps with API rate limits if your func calls Enrichr internally
#             time.sleep(0.03)

#         # Keep only robust terms
#         for t, c in counts.items():
#             freq = c / max(R, 1)
#             if freq >= recurrence_cutoff:
#                 if '|' in t:
#                     term, gene_set = t.split('|', 1)
#                     rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
#                 else:
#                     rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

#     return (pd.DataFrame(rows)
#               .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
#               .reset_index(drop=True))

In [ ]:
# twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
#                                 R=25, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
# twr3

In [ ]:
# terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
#                                 R=10, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
# terms_with_recurrence

In [ ]:
# # rename important terms to match terms_with_recurrence
# important_terms = important_terms.rename(columns={'index': 'community_id'})
# important_terms = important_terms.rename(columns={'Term': 'term'})

In [ ]:
# terms_with_rec_merged = important_terms.merge(
#     terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
#     on=['community_id', 'term', 'Gene_set'],
#     how='left'
# )

# terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

# terms_with_rec_merged = terms_with_rec_merged.sort_values(
#     ['community_id', 'recurrence'],
#     ascending=[True, False]
# ).reset_index(drop=True)

In [ ]:
# terms_with_rec_merged

In [ ]:
# community_summary = (
#     terms_with_rec_merged
#     .groupby("community_id")["recurrence"]
#     .agg(mean_recurrence="mean", term_count="count")
#     .reset_index()
# )

# print(community_summary)

In [ ]:
# display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [ ]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [ ]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))